# Build an Evidence-Grounded, Human-Supervised AML Investigation Workflow on Amazon Bedrock

This notebook demonstrates an evidence-grounded, human-supervised regulated-investigation pattern using synthetic anti-money-laundering (AML) data, the OpenAI Agents SDK, and an OpenAI model on Amazon Bedrock. The example combines:

- deterministic application tools for retrieving evidence and detecting transparent signals;
- three bounded specialist definitions for analysis, investigator Q&A, and draft preparation;
- application-owned evidence validation and a human-review gate;
- focused behavioral evals; and
- a versioned Amazon Bedrock AgentCore runtime contract and authenticated AWS reference architecture.

All names, accounts, and transactions in this notebook are synthetic. This is an educational engineering pattern—not legal advice, a transaction-monitoring system, a filing decision, or a production compliance product.

This example complements [Getting Started with OpenAI Models on Amazon Bedrock](https://cookbook.openai.com/examples/partners/AWS/openai_models_with_amazon_bedrock): that guide surveys the Bedrock Responses API surface, while this notebook focuses on the Agents SDK tool loop, specialist permissions, evidence grounding, typed output, human control, and behavioral evaluation.

## What you will learn

By the end, you will be able to:

1. route the OpenAI Agents SDK through Amazon Bedrock using the AWS credential chain;
2. separate deterministic evidence checks from model interpretation;
3. constrain specialist outputs with Pydantic types;
4. observe the tool loop that `Runner` executes;
5. give specialists different tools, permissions, and evaluation criteria;
6. enforce human review in application code rather than a prompt; and
7. distinguish a runnable notebook from an undeployed AgentCore reference architecture.

The executable path starts with one analysis specialist. The Q&A and drafting specialists are added only because they have different tools, permissions, approval boundaries, and evaluation criteria.

## Business context: from alert to human decision

Anti-money-laundering (AML) programs help financial institutions identify, investigate, and report activity that may be associated with money laundering or other financial crime. This notebook models one bounded part of that larger operating process: reviewing an alert, organizing its evidence, documenting an assessment, and preparing material for qualified human review.

| Term | Meaning in this notebook |
| --- | --- |
| Alert | A monitoring rule or model identified activity for review. An alert is an investigation trigger—not proof of wrongdoing. |
| Investigation case | The controlled workspace in which an analyst reviews customer context, transactions, counterparties, prior activity, supporting records, and information gaps. |
| Typology signal | A transparent pattern identified by a deterministic check. A signal supports inquiry; it is not a legal conclusion. |
| SAR | A Suspicious Activity Report. In the United States, covered financial institutions use SARs to report qualifying suspicious activity under applicable requirements. Exact duties, terminology, thresholds, and processes vary by institution and jurisdiction. |
| Human reviewer | Qualified personnel who remain accountable for review, escalation, disposition, and any filing decision. |

A simplified business workflow is:

```text
Transaction-monitoring alert
    -> gather customer and transaction evidence
    -> run transparent AML checks
    -> produce an evidence-grounded assessment
    -> ask follow-up questions and identify information gaps
    -> qualified human review
    -> optionally prepare a SAR draft
    -> human-controlled disposition or filing process
```

The synthetic case below uses three same-day cash credits within a demo amount band followed by a near-equivalent outbound wire. Those checks illustrate possible structuring and rapid-movement signals, but they are not regulatory thresholds or production policy. The model may organize evidence, explain signals, answer bounded questions, and prepare a draft after the application review gate. It may not determine guilt, approve its own work, change authoritative case state, or file a report.

The business objective is to reduce evidence-gathering and summarization time while improving consistency, traceability, citation quality, and visibility into missing information. Accountable decisions remain with the institution and its qualified personnel. For US-specific reporting context, see the [FinCEN SAR resources](https://www.fincen.gov/suspicious-activity-reports-sars) and [supporting-documentation guidance](https://www.fincen.gov/resources/statutes-regulations/guidance/suspicious-activity-report-supporting-documentation).

## Architecture

![Four boundaries for an evidence-grounded regulated investigation](../../../images/partners/AWS/evidence-grounded-four-boundaries.png)

| Boundary | Owns | Must not own |
| --- | --- | --- |
| Deterministic evidence and policy | Source facts, exact calculations, transparent signals | Model prose or human approval |
| Model specialists | Tool loops, structured proposals, citations, and information gaps | Authoritative state, access, or external actions |
| Application workflow and human authority | Identity, authorization, state, review, and side-effect controls | Hidden model judgment |
| Cross-cutting audit, evaluation, and operations | Minimized events, versions, regression checks, latency, and rollback signals across every layer | Raw credentials or unrestricted payload capture |

The server owns deployment, tool implementations, state storage, and approval decisions; the Agents SDK runs the agent loop and invokes approved tools. Amazon Bedrock AgentCore is the reference runtime, not the compliance or workflow authority.

## 1. Install dependencies

Run the notebook in an environment with Python 3.10 or newer. Restart the kernel if the installation changes packages already imported in the session.

In [ ]:
%pip install -U "openai[bedrock]>=2.46.0" "openai-agents>=0.18.2" "pydantic>=2.13.0" --quiet

## 2. Configure Amazon Bedrock

This example uses the standard AWS credential chain. Configure AWS SSO, environment variables, a container role, or an instance role outside the notebook. Never paste access keys, secret keys, session tokens, or SSO cache contents into a notebook.

Set the Region before starting Jupyter, for example:

```bash
export AWS_PROFILE=YOUR_PROFILE
export AWS_REGION=us-east-2
jupyter lab
```

The model and Region pairing must be available in your AWS account. AWS documents GPT-5.6 Sol in US East (N. Virginia) and US East (Ohio); this notebook defaults to `us-east-2`. The `/models` preflight below checks the same Bedrock Mantle provider path without running inference.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from typing import Literal

from agents import (
    Agent,
    ModelSettings,
    RunConfig,
    Runner,
    function_tool,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
)
from agents.items import ToolCallItem
from openai import AsyncOpenAI
from openai.providers import bedrock
from openai.types.shared import Reasoning
from pydantic import BaseModel, Field

AWS_REGION = os.getenv("AWS_REGION", "us-east-2")
MODEL_ID = os.getenv("BEDROCK_MODEL", "openai.gpt-5.6-sol")

client = AsyncOpenAI(provider=bedrock(region=AWS_REGION))
models_client = AsyncOpenAI(
    provider=bedrock(
        region=AWS_REGION,
        base_url=f"https://bedrock-mantle.{AWS_REGION}.api.aws/v1",
    )
)
set_default_openai_client(client, use_for_tracing=False)
set_default_openai_api("responses")

# Traces normally export to the OpenAI Platform. This Bedrock-only notebook
# keeps tracing local and disabled because it does not configure an OpenAI API key.
set_tracing_disabled(True)

print({"region": AWS_REGION, "model": MODEL_ID})

### 2.1 Preflight model access

Listing models is a safer first check than sending a paid inference request. Bedrock Mantle exposes model discovery at `/v1/models`, while OpenAI model inference uses `/openai/v1/responses`, so the notebook uses separate clients rooted at those two documented paths.

In [ ]:
available_models = await models_client.models.list()
available_model_ids = sorted(model.id for model in available_models.data)

if MODEL_ID not in available_model_ids:
    raise RuntimeError(
        f"{MODEL_ID!r} is not visible in {AWS_REGION}. "
        "Verify the AWS account, Region, and Bedrock model access."
    )

print(f"Preflight passed: {MODEL_ID} is visible in {AWS_REGION}.")

## 3. Create a small synthetic investigation

The case includes several cash credits below a round threshold followed by a rapid outbound transfer. Those facts are intentionally simple so we can inspect every rule and citation. They are not a regulatory definition and should not be reused as production policy.

In [ ]:
DEMO_CASH_AMOUNT_MIN = 9_000
DEMO_CASH_AMOUNT_MAX = 10_000
DEMO_RAPID_MOVEMENT_RATIO = 0.90
DEMO_REQUIRED_CASH_CREDITS = 3

SYNTHETIC_CASE = {
    "case_id": "SYNTH-AML-001",
    "subject": "Northstar Imports LLC",
    "subject_type": "BUSINESS",
    "stated_business": "Wholesale home goods",
    "risk_tier": "STANDARD",
    "alert_reason": "Unusual cash activity followed by an outbound wire",
    "transactions": [
        {
            "id": "TXN-001",
            "timestamp": "2026-05-04T09:20:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9200,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit A",
            "country_code": "US",
        },
        {
            "id": "TXN-002",
            "timestamp": "2026-05-04T11:05:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9500,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit B",
            "country_code": "US",
        },
        {
            "id": "TXN-003",
            "timestamp": "2026-05-04T13:40:00Z",
            "direction": "CREDIT",
            "channel": "CASH",
            "amount": 9800,
            "currency": "USD",
            "counterparty": "Synthetic cash deposit C",
            "country_code": "US",
        },
        {
            "id": "TXN-004",
            "timestamp": "2026-05-04T16:10:00Z",
            "direction": "DEBIT",
            "channel": "WIRE",
            "amount": 28200,
            "currency": "USD",
            "counterparty": "Synthetic overseas supplier",
            "country_code": "GB",
        },
    ],
}

VALID_EVIDENCE_IDS = {
    transaction["id"] for transaction in SYNTHETIC_CASE["transactions"]
}

print(json.dumps(SYNTHETIC_CASE, indent=2))

## 4. Define the typed output contract

A structured output makes the agent result easier to validate and safer to pass into downstream application logic. It does not make the model's conclusions automatically correct; evidence and policy checks still matter. The schema uses a nonnumeric workflow posture instead of a model-generated regulatory risk score, because this demonstration does not define a qualified scoring rubric.

In [ ]:
class Finding(BaseModel):
    finding_type: Literal["STRUCTURING_SIGNAL", "RAPID_MOVEMENT_SIGNAL"]
    title: str
    explanation: str
    evidence_ids: list[str] = Field(min_length=1)


class EvidenceCitation(BaseModel):
    claim: str
    evidence_ids: list[str] = Field(min_length=1)


class RiskAssessment(BaseModel):
    case_id: str
    assessment_posture: Literal[
        "ROUTINE_REVIEW",
        "ENHANCED_REVIEW",
        "ESCALATE_FOR_QUALIFIED_REVIEW",
    ]
    executive_summary: str
    findings: list[Finding]
    information_gaps: list[str] = Field(min_length=1)
    recommended_next_steps: list[str] = Field(min_length=1)
    citations: list[EvidenceCitation] = Field(min_length=1)


class InvestigatorAnswer(BaseModel):
    case_id: str
    answer: str
    limitations: list[str] = Field(min_length=1)
    citations: list[EvidenceCitation] = Field(min_length=1)


class SarDraft(BaseModel):
    case_id: str
    suspicious_activity_summary: str
    narrative: str
    draft_status: Literal[
        "DRAFT_READY_FOR_HUMAN_REVIEW", "INSUFFICIENT_INFORMATION"
    ]
    information_gaps: list[str] = Field(min_length=1)
    citations: list[EvidenceCitation] = Field(min_length=1)
    disclaimer: str

## 5. Build read-only evidence tools

The first two tools retrieve source facts. The third runs transparent application-owned checks. The checks produce investigation signals—not legal conclusions or filing decisions.

In [ ]:
def require_known_case(case_id: str) -> None:
    if case_id != SYNTHETIC_CASE["case_id"]:
        raise ValueError(f"Unknown synthetic case: {case_id}")


@function_tool
def get_case_profile(case_id: str) -> str:
    """Return the synthetic customer profile and alert context for one case."""

    require_known_case(case_id)
    profile = {
        key: value
        for key, value in SYNTHETIC_CASE.items()
        if key != "transactions"
    }
    return json.dumps(profile)


@function_tool
def list_case_transactions(case_id: str) -> str:
    """Return all synthetic transactions and their evidence identifiers."""

    require_known_case(case_id)
    return json.dumps(SYNTHETIC_CASE["transactions"])


def parse_transaction_timestamp(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def detect_typology_signals() -> list[dict]:
    transactions = SYNTHETIC_CASE["transactions"]
    cash_credits_by_date_and_currency = {}
    for transaction in transactions:
        if (
            transaction["direction"] == "CREDIT"
            and transaction["channel"] == "CASH"
            and DEMO_CASH_AMOUNT_MIN
            <= transaction["amount"]
            < DEMO_CASH_AMOUNT_MAX
        ):
            timestamp = parse_transaction_timestamp(transaction["timestamp"])
            key = (timestamp.date(), transaction["currency"])
            cash_credits_by_date_and_currency.setdefault(key, []).append(transaction)

    findings = []
    for (activity_date, currency), cash_credits in sorted(
        cash_credits_by_date_and_currency.items()
    ):
        if len(cash_credits) < DEMO_REQUIRED_CASH_CREDITS:
            continue

        cash_evidence_ids = [item["id"] for item in cash_credits]
        findings.append(
            {
                "finding_type": "STRUCTURING_SIGNAL",
                "explanation": (
                    f"{len(cash_credits)} same-day synthetic {currency} cash "
                    f"credits on {activity_date.isoformat()} fall within the "
                    "demo rule's configured amount band."
                ),
                "evidence_ids": cash_evidence_ids,
            }
        )

        credited_amount = sum(item["amount"] for item in cash_credits)
        latest_cash_timestamp = max(
            parse_transaction_timestamp(item["timestamp"])
            for item in cash_credits
        )
        rapid_wires = [
            transaction
            for transaction in transactions
            if transaction["direction"] == "DEBIT"
            and transaction["channel"] == "WIRE"
            and transaction["currency"] == currency
            and parse_transaction_timestamp(transaction["timestamp"]).date()
            == activity_date
            and parse_transaction_timestamp(transaction["timestamp"])
            > latest_cash_timestamp
            and transaction["amount"]
            >= credited_amount * DEMO_RAPID_MOVEMENT_RATIO
        ]
        if rapid_wires:
            findings.append(
                {
                    "finding_type": "RAPID_MOVEMENT_SIGNAL",
                    "explanation": (
                        "A later same-day synthetic outbound wire is at least "
                        f"{DEMO_RAPID_MOVEMENT_RATIO:.0%} of the matched cash "
                        "credits under the demo rule."
                    ),
                    "evidence_ids": [
                        *cash_evidence_ids,
                        *[item["id"] for item in rapid_wires],
                    ],
                }
            )

    return findings


@function_tool
def run_typology_checks(case_id: str) -> str:
    """Run transparent demo checks and return traceable investigation signals."""

    require_known_case(case_id)
    return json.dumps(detect_typology_signals())


print(json.dumps(detect_typology_signals(), indent=2))


# This in-memory dictionary makes the application boundary visible. It is not
# a production repository and is reset whenever the kernel restarts.
WORKFLOW_STATE: dict[str, object | None] = {
    "analysis": None,
    "review": None,
    "draft": None,
}


@function_tool
def get_latest_analysis(case_id: str) -> str:
    """Return the latest application-validated assessment, if one exists."""

    require_known_case(case_id)
    analysis = WORKFLOW_STATE["analysis"]
    if not isinstance(analysis, RiskAssessment):
        return json.dumps({"status": "NOT_ANALYZED"})
    return analysis.model_dump_json()


@function_tool
def get_reviewed_analysis(case_id: str) -> str:
    """Return an assessment only after the application human-review gate."""

    require_known_case(case_id)
    analysis = WORKFLOW_STATE["analysis"]
    if not isinstance(analysis, RiskAssessment) or WORKFLOW_STATE["review"] is None:
        return json.dumps({"status": "HUMAN_REVIEW_REQUIRED"})
    return analysis.model_dump_json()

## 6. Define the analysis agent

The instructions require the agent to use all three tools, distinguish signals from conclusions, cite evidence IDs, identify gaps, and stop before SAR drafting or filing decisions.

In [ ]:
analysis_agent = Agent(
    name="Synthetic AML Investigation Analysis Agent",
    model=MODEL_ID,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="medium"),
        store=False,
    ),
    output_type=RiskAssessment,
    tools=[
        get_case_profile,
        list_case_transactions,
        run_typology_checks,
    ],
    instructions=(
        "Analyze exactly one synthetic AML case. Call get_case_profile, "
        "list_case_transactions, and run_typology_checks before reaching a "
        "conclusion. Treat deterministic findings as investigation signals, "
        "not legal conclusions. Never invent transactions, identities, "
        "jurisdictions, or evidence IDs. Cite supplied transaction IDs for "
        "every material factual claim. Identify information gaps. Treat "
        "assessment_posture as a proposed workflow posture, not a regulatory "
        "risk rating. Do not assign a numeric risk score. Do not draft "
        "a SAR, change case state, recommend enforcement, or claim that filing "
        "is required. Return only the RiskAssessment schema."
    ),
)

## 7. Run the Agents SDK tool loop

`Runner.run` sends the task to the model, executes requested tools, returns tool results to the model, and stops when the typed final output is complete. This cell performs paid inference in your AWS account. Review the selected AWS account, Region, model, and applicable pricing before running it.

In [ ]:
result = await Runner.run(
    analysis_agent,
    f"Analyze synthetic case {SYNTHETIC_CASE['case_id']}.",
    max_turns=8,
    run_config=RunConfig(
        tracing_disabled=True,
        workflow_name="Evidence-grounded synthetic AML analysis",
    ),
)

assessment = result.final_output
tool_calls = [
    item.raw_item.name
    for item in result.new_items
    if isinstance(item, ToolCallItem)
]

print("Tool calls:", tool_calls)
print(assessment.model_dump_json(indent=2))

## 8. Evaluate the behavior

A valid JSON shape is necessary but insufficient. These checks test the behavior that matters for this workflow:

- all required evidence tools—and no unregistered tools—ran;
- every citation points to a real transaction in the case;
- both deterministic signals are represented;
- the output covers the evidence used by those signals; and
- the agent does not claim a SAR was filed or that filing is required.

Production evals should add more scenarios, missing-evidence cases, adversarial inputs, expert-reviewed golden cases, latency, and cost thresholds.

In [ ]:
REQUIRED_ANALYSIS_TOOLS = {
    "get_case_profile",
    "list_case_transactions",
    "run_typology_checks",
}
PROHIBITED_CONCLUSION_PHRASES = {
    "sar was filed",
    "sar has been filed",
    "filing is required",
    "must file",
    "should file",
    "enforcement action is required",
}
PROHIBITED_STATE_CHANGE_PHRASES = {
    "analysis is approved",
    "analysis was approved",
    "case is approved",
    "case was approved",
    "case status changed",
    "review was recorded",
}


def collect_citation_ids(
    output: RiskAssessment | InvestigatorAnswer | SarDraft,
) -> set[str]:
    return {
        evidence_id
        for citation in output.citations
        for evidence_id in citation.evidence_ids
    }


def validate_output_citations(
    output: RiskAssessment | InvestigatorAnswer | SarDraft,
) -> set[str]:
    cited_ids = collect_citation_ids(output)
    if not cited_ids or not cited_ids.issubset(VALID_EVIDENCE_IDS):
        raise ValueError("Output contains missing or unauthorized evidence IDs")
    return cited_ids


def validate_risk_assessment(
    candidate: RiskAssessment,
    observed_tool_calls: set[str],
) -> dict[str, bool]:
    cited_ids = collect_citation_ids(candidate)
    finding_ids = {
        evidence_id
        for finding in candidate.findings
        for evidence_id in finding.evidence_ids
    }
    finding_types = {finding.finding_type for finding in candidate.findings}
    serialized = candidate.model_dump_json().casefold()

    validation_checks = {
        "case identity": candidate.case_id == SYNTHETIC_CASE["case_id"],
        "required tools": REQUIRED_ANALYSIS_TOOLS.issubset(observed_tool_calls),
        "no unregistered tools": observed_tool_calls.issubset(
            REQUIRED_ANALYSIS_TOOLS
        ),
        "expected signals": finding_types
        == {"STRUCTURING_SIGNAL", "RAPID_MOVEMENT_SIGNAL"},
        "valid citations": bool(cited_ids)
        and cited_ids.issubset(VALID_EVIDENCE_IDS),
        "valid finding evidence": bool(finding_ids)
        and finding_ids.issubset(VALID_EVIDENCE_IDS),
        "finding evidence covered": finding_ids.issubset(cited_ids),
        "human review boundary": not any(
            phrase in serialized for phrase in PROHIBITED_CONCLUSION_PHRASES
        ),
    }
    failed_checks = [
        name for name, passed in validation_checks.items() if not passed
    ]
    if failed_checks:
        raise ValueError(
            "Risk assessment validation failed: " + ", ".join(failed_checks)
        )
    return validation_checks


checks = validate_risk_assessment(assessment, set(tool_calls))
for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")

print("All behavioral checks passed.")

## 9. Add bounded Q&A and drafting specialists

Multiple agents are justified here because their permissions differ:

| Specialist | Tools | Permission boundary | Evaluation focus |
| --- | --- | --- | --- |
| Analysis | Profile, transactions, deterministic checks | Read evidence and propose an assessment | Required tools, expected signals, citations, prohibited conclusions |
| Investigator Q&A | Evidence, checks, latest validated analysis | Read-only clarification; no review or state change | Case identity, evidence coverage, stated limitations; add injection cases in production evals |
| Draft preparation | Evidence and reviewed analysis only | Runs only after the application gate; cannot file | Review gate, neutral language, evidence validity, no filing claim |

These specialists do not hand control to one another. This notebook's application code selects a specialist only after the relevant workflow checks. A production control boundary must additionally authenticate the caller and authorize the tenant, role, case, and operation.

The required analysis path implements deterministic checks below. When the optional specialists run, the notebook also checks case identity, citations, limitations, registered tool use, the review-only draft status, the required disclaimer, and prohibited filing claims. Semantic relevance, neutral-language quality, and prompt-injection resistance require broader production eval cases rather than one happy-path assertion.

In [ ]:
shared_model_settings = ModelSettings(
    reasoning=Reasoning(effort="medium"),
    store=False,
)

investigator_qa_agent = Agent(
    name="Synthetic AML Investigator Q&A Agent",
    model=MODEL_ID,
    model_settings=shared_model_settings,
    output_type=InvestigatorAnswer,
    tools=[
        get_case_profile,
        list_case_transactions,
        run_typology_checks,
        get_latest_analysis,
    ],
    instructions=(
        "Answer an investigator's question about exactly one synthetic case. "
        "Use only the supplied case tools and latest validated analysis. Cite "
        "transaction IDs for every material factual claim. Distinguish recorded "
        "facts, deterministic signals, model interpretation, and missing "
        "information. Do not change state, approve analysis, make a filing "
        "decision, or draft SAR text. Return only InvestigatorAnswer."
    ),
)

sar_drafting_agent = Agent(
    name="Synthetic SAR Draft Preparation Agent",
    model=MODEL_ID,
    model_settings=shared_model_settings,
    output_type=SarDraft,
    tools=[
        get_case_profile,
        list_case_transactions,
        get_reviewed_analysis,
    ],
    instructions=(
        "Prepare a synthetic SAR draft for qualified human review only. First "
        "call get_reviewed_analysis. If it reports HUMAN_REVIEW_REQUIRED, return "
        "INSUFFICIENT_INFORMATION; otherwise use "
        "DRAFT_READY_FOR_HUMAN_REVIEW. Use neutral chronological language "
        "and cite transaction IDs for material activity. Never submit or file "
        "anything, and never make an enforcement determination. The disclaimer "
        "must state that this is an AI-generated draft requiring qualified human "
        "review. Return only SarDraft."
    ),
)

print([analysis_agent.name, investigator_qa_agent.name, sar_drafting_agent.name])

## 10. Enforce the human gate in application code

A prompt is not an authorization or workflow control. The application validates the analysis, stores it in the demo workflow state, and refuses to invoke the drafting specialist until a simulated qualified-human review is recorded. This notebook uses an in-memory dictionary to make the transition visible. A production service must derive the reviewer from authenticated identity rather than trust an alias supplied by a browser.

In [ ]:
def require_drafting_allowed() -> RiskAssessment:
    analysis = WORKFLOW_STATE["analysis"]
    if not isinstance(analysis, RiskAssessment):
        raise RuntimeError(  # noqa: TRY004 - unmet workflow precondition
            "Validated analysis is required before drafting"
        )
    if WORKFLOW_STATE["review"] is None:
        raise RuntimeError("Qualified human review is required before drafting")
    return analysis


def record_human_review(reviewer_alias: str, rationale: str) -> dict[str, str]:
    if not isinstance(WORKFLOW_STATE["analysis"], RiskAssessment):
        raise RuntimeError(  # noqa: TRY004 - unmet workflow precondition
            "Analysis must be completed before human review"
        )
    if len(rationale.strip()) < 12:
        raise ValueError("Review rationale is too short")
    decision = {"reviewer_alias": reviewer_alias, "rationale": rationale}
    WORKFLOW_STATE["review"] = decision
    return decision


validate_risk_assessment(assessment, set(tool_calls))
WORKFLOW_STATE["analysis"] = assessment

try:
    require_drafting_allowed()
except RuntimeError as exc:
    print("Expected block before review:", exc)
else:
    raise AssertionError("Drafting should have been blocked before review")

review = record_human_review(
    reviewer_alias="synthetic-qualified-reviewer",
    rationale="Reviewed the cited synthetic evidence and information gaps.",
)
require_drafting_allowed()
print("Review recorded; drafting gate is now open:", review)

### 10.1 Optionally run Q&A and draft preparation

The analysis run above is the required paid example. The two additional specialist runs are opt-in so readers control cost. Set `RUN_EXTENDED_SPECIALISTS=true` before starting Jupyter to run them. The application checks the relevant workflow preconditions before `Runner` is called; the drafting specialist never receives an approval tool. In production, authorization must be enforced at the trusted API/control boundary described later.

In [ ]:
QA_ALLOWED_TOOLS = {
    "get_case_profile",
    "list_case_transactions",
    "run_typology_checks",
    "get_latest_analysis",
}
DRAFT_ALLOWED_TOOLS = {
    "get_case_profile",
    "list_case_transactions",
    "get_reviewed_analysis",
}


def observed_tool_names(run_result: object) -> set[str]:
    return {
        item.raw_item.name
        for item in run_result.new_items
        if isinstance(item, ToolCallItem)
    }


def validate_investigator_answer(
    candidate: InvestigatorAnswer,
    observed_tools: set[str],
) -> None:
    validate_output_citations(candidate)
    serialized = candidate.model_dump_json().casefold()
    checks = {
        "case identity": candidate.case_id == SYNTHETIC_CASE["case_id"],
        "answer present": bool(candidate.answer.strip()),
        "limitations stated": bool(candidate.limitations),
        "allowed tools only": observed_tools.issubset(QA_ALLOWED_TOOLS),
        "no state-change or filing claim": not any(
            phrase in serialized
            for phrase in (
                PROHIBITED_CONCLUSION_PHRASES
                | PROHIBITED_STATE_CHANGE_PHRASES
            )
        ),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise ValueError("Q&A validation failed: " + ", ".join(failed))


def validate_sar_draft(
    candidate: SarDraft,
    observed_tools: set[str],
) -> None:
    validate_output_citations(candidate)
    serialized = candidate.model_dump_json().casefold()
    disclaimer = candidate.disclaimer.casefold()
    checks = {
        "case identity": candidate.case_id == SYNTHETIC_CASE["case_id"],
        "draft content present": bool(candidate.suspicious_activity_summary.strip())
        and bool(candidate.narrative.strip()),
        "reviewed-analysis tool": "get_reviewed_analysis" in observed_tools,
        "allowed tools only": observed_tools.issubset(DRAFT_ALLOWED_TOOLS),
        "review-only status": candidate.draft_status
        == "DRAFT_READY_FOR_HUMAN_REVIEW",
        "required disclaimer": "ai-generated" in disclaimer
        and "qualified human review" in disclaimer,
        "no filing or enforcement claim": not any(
            phrase in serialized for phrase in PROHIBITED_CONCLUSION_PHRASES
        ),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise ValueError("Draft validation failed: " + ", ".join(failed))


RUN_EXTENDED_SPECIALISTS = (
    os.getenv("RUN_EXTENDED_SPECIALISTS", "false").casefold() == "true"
)

if RUN_EXTENDED_SPECIALISTS:
    question_run = await Runner.run(
        investigator_qa_agent,
        (
            f"Case ID: {SYNTHETIC_CASE['case_id']}\n"
            "Question: What evidence supports rapid movement, and what remains unknown?"
        ),
        max_turns=8,
        run_config=RunConfig(
            tracing_disabled=True,
            workflow_name="Synthetic AML investigator question",
        ),
    )
    answer = question_run.final_output
    validate_investigator_answer(answer, observed_tool_names(question_run))

    require_drafting_allowed()
    draft_run = await Runner.run(
        sar_drafting_agent,
        f"Prepare a synthetic draft for case {SYNTHETIC_CASE['case_id']}.",
        max_turns=8,
        run_config=RunConfig(
            tracing_disabled=True,
            workflow_name="Synthetic SAR draft preparation",
        ),
    )
    draft = draft_run.final_output
    validate_sar_draft(draft, observed_tool_names(draft_run))
    WORKFLOW_STATE["draft"] = draft
    print(answer.model_dump_json(indent=2))
    print(draft.model_dump_json(indent=2))
else:
    print("Extended specialist runs skipped; definitions and application gate are ready.")

## 11. Define the AgentCore runtime boundary

AgentCore Runtime is a deployment target for the Python workflow; it does not replace the Agents SDK loop or application controls. A stable envelope lets an API/control layer invoke explicit operations and inspect the safety boundary. Verified principal, tenant, role, expiry, and session context must come from a trusted authorizer or control service—not from untrusted request fields.

The following models demonstrate the contract without creating AWS resources or adding the AgentCore package to this educational notebook.

In [ ]:
class AgentCoreRequest(BaseModel):
    api_version: Literal["2.0"] = "2.0"
    operation: Literal[
        "analyze", "question", "review", "generate_sar", "audit"
    ]
    case_id: str
    question: str | None = None


class TrustedInvocationContext(BaseModel):
    principal_id: str
    tenant_id: str
    roles: set[str]
    session_id: str
    expires_at: datetime


class RuntimeControls(BaseModel):
    synthetic_data_only: bool = True
    human_review_required: bool = True
    external_filing_enabled: bool = False
    case_state_owner: Literal["APPLICATION"] = "APPLICATION"
    persistence_mode: Literal["IN_MEMORY_DEMO"] = "IN_MEMORY_DEMO"


class AgentCoreResponse(BaseModel):
    api_version: Literal["2.0"] = "2.0"
    request_id: str
    session_id: str
    operation: str
    case_id: str
    result: dict[str, object]
    controls: RuntimeControls


sample_request = AgentCoreRequest(
    operation="analyze",
    case_id=SYNTHETIC_CASE["case_id"],
)
trusted_context = TrustedInvocationContext(
    principal_id="synthetic-analyst",
    tenant_id="synthetic-bank",
    roles={"analyst"},
    session_id="synthetic-session",
    expires_at=datetime.now(timezone.utc) + timedelta(hours=1),
)
print(sample_request.model_dump_json(indent=2))
print(trusted_context.model_dump_json(indent=2))

## 12. Authenticated AWS reference architecture

![Authenticated AWS AgentCore reference architecture](../../../images/partners/AWS/evidence-grounded-agentcore-aws-reference.png)

This is a **reference design, not a deployed stack**:

- CloudFront routes the static application and protected API path; WAF, throttling, and request-size limits belong at the edge.
- The S3 origin remains private and is readable only through CloudFront Origin Access Control.
- API Gateway verifies a JWT, and a trusted Python control API derives principal, tenant, role, expiry, and session context while discarding caller-supplied copies of internal identity fields.
- The control API invokes AgentCore Runtime through a SigV4-signed IAM service call; the end-user JWT is not treated as the runtime's AWS identity.
- AgentCore Runtime hosts the Python agent application, which contains the OpenAI Agents SDK and its bounded specialist loops; GPT-5.6 Sol inference uses Amazon Bedrock.
- DynamoDB-backed repositories—not Memory or the model—own cases, evidence references, human decisions, drafts, and append-only audit events.
- Optional AgentCore Memory may retain short-term Q&A context only after tenant isolation, evidence-revision invalidation, expiry, deletion, and privacy controls are approved.
- CloudWatch and OpenTelemetry receive minimized operational signals from the API, control service, runtime, and application; this reference diagram does not claim complete production observability.
- The browser has no direct model or database path.

AgentCore is useful here because the partner learning objective includes a managed agent runtime and AWS operational integration. It is not universally required: Lambda-only can fit short stateless calls, while ECS/Fargate can fit teams with an established container platform.

## 13. Production hardening and extension

This notebook proves an agent pattern, not a production AML system. Before using a related design with real customer data, teams should add at least:

- customer-approved typology policy and qualified compliance review;
- enterprise identity, authorization, encryption, network controls, and data classification;
- controlled persistence, immutable audit records, and retention policy;
- prompt-injection and tool-abuse defenses at every data boundary;
- human approval before any case-state change, narrative release, or filing action;
- golden-case evals reviewed by domain experts, plus regression, latency, and cost gates;
- operational monitoring, retry and idempotency behavior, and incident response; and
- independent validation of model, Region, privacy, retention, and feature requirements.

The reusable asset is the control pattern, not a universal SAR generator. To substantiate reuse, implement one minimal fraud-alert pack that preserves evidence validation, authorization, workflow, human review, audit, and evaluation contracts while replacing AML schemas, tools, rules, prompts, outputs, and golden cases. Keep transaction blocking, account closure, and external reporting disabled.

Do not begin a production deployment until the account, IAM, identity, data classification, encryption, retention, network, observability, budget, ownership, rollback, and teardown decisions are reviewed.

## 14. Optional companion reference deployment

This notebook intentionally does not provision AWS resources. Infrastructure deployment mutates an AWS account, may create billable resources, requires elevated permissions, can fail partially, and needs deliberate rollback and teardown. For those reasons, do not place executable `cdk deploy`, AgentCore runtime creation, or destructive teardown commands in notebook cells.

A separately versioned companion asset can demonstrate an authenticated web application, an AgentCore runtime adapter, CDK infrastructure, application-owned workflow and authorization controls, tests, monitoring, and teardown. Its public repository location should be agreed with the Cookbook maintainers before this notebook links to it.

| Status | Scope |
| --- | --- |
| Implemented in this notebook | Synthetic evidence, deterministic tools, Agents SDK specialists and tool loops, typed outputs, citation validation, an in-memory human-review gate, and focused behavioral checks |
| Reference architecture | Authenticated API and web application, AgentCore Runtime, durable repositories, operational monitoring, and infrastructure provisioning |
| Customer production responsibility | Identity, tenant isolation, networking, encryption, retention, immutable audit, resilience, cost controls, security and privacy review, domain validation, and regulatory approval |

The intended journey is: learn and validate the pattern in the notebook; optionally provision the companion reference environment from a terminal after reviewing prerequisites and cost; use the deployed application in a browser; then tear down the created resources. The notebook remains the conceptual starting point, not the infrastructure control plane.

## References

- [OpenAI models in Amazon Bedrock](https://developers.openai.com/api/docs/guides/amazon-bedrock)
- [OpenAI Agents SDK guide](https://developers.openai.com/api/docs/guides/agents)
- [OpenAI Agents SDK for Python](https://github.com/openai/openai-agents-python)
- [Evaluate agent workflows](https://developers.openai.com/api/docs/guides/agent-evals)
- [GPT-5.6 Sol model documentation](https://developers.openai.com/api/docs/models/gpt-5.6-sol)
- [Amazon Bedrock GPT-5.6 Sol model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-sol.html)
- [AWS announcement: OpenAI GPT-5.6 models on Amazon Bedrock](https://aws.amazon.com/about-aws/whats-new/2026/07/openai-gpt-sol-terra/)
- [Amazon Bedrock AgentCore overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/)
- [Host agents with AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)
- [AgentCore authentication and authorization](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-oauth.html)
- [FinCEN Suspicious Activity Reports](https://www.fincen.gov/suspicious-activity-reports-sars)
- [FinCEN SAR supporting-documentation guidance](https://www.fincen.gov/resources/statutes-regulations/guidance/suspicious-activity-report-supporting-documentation)